In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
NVIDIA GeForce RTX 2050


In [2]:
import os

# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import BaseChronosPipeline, Chronos2Pipeline

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
chronos2: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2", device_map="cuda"
)

In [3]:
from pprint import pprint
from copy import deepcopy

import pandas as pd

from utilsforecast.losses import mase
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive

from src.meta.arima._data_reader import ModelIO
from src.chronos_data import ChronosDataset

OVERRIDE_DS = False
algorithm = "catboost"
source = "m4_monthly"
FILENAME = f"assets/trained_metaarima_{source}_{algorithm}.joblib.gz"
meta_arima = ModelIO.load_model(FILENAME)

target = "monash_tourism_monthly"

df, horizon, _, freq, seas_len = ChronosDataset.load_everything(target)
train, test = ChronosDataset.time_wise_split(df, horizon)

sf_models = [AutoARIMA(season_length=seas_len), SeasonalNaive(season_length=seas_len)]

#  TODO ADD TSFM'S NAMES TO LIST
model_names = ["MetaARIMA", "AutoARIMA", "SeasonalNaive", "Chronos2"]

uids = train["unique_id"].unique().tolist()

results, predictions = [], []
for uid in uids:
    print(uid)
    # uid="T000055"

    df_uid_tr = train.query(f'unique_id=="{uid}"').reset_index(drop=True)
    df_uid_ts = test.query(f'unique_id=="{uid}"').reset_index(drop=True)
    if df_uid_ts.isna().any()["y"]:
        continue

    meta_arima.fit(df_uid_tr, freq=freq, seas_length=seas_len)

    fcst_ma = meta_arima.predict(h=horizon)

    sf = StatsForecast(models=deepcopy(sf_models), freq=freq)
    sf.fit(df_uid_tr)

    fcst_aa = sf.forecast(h=horizon)

    # TODO tsfm inference
    fcst_tsfm1 = chronos2.predict_df(df_uid_tr, prediction_length=horizon, id_column="unique_id", timestamp_column="ds", target="y")

    fcst_tsfm1 = fcst_tsfm1.rename(columns={"predictions": "Chronos2"})
    fcst_tsfm1 = fcst_tsfm1[["unique_id", "ds", "Chronos2"]]


    # TODO transform to structure like fcst_aa

    if OVERRIDE_DS:
        fcst_ma["ds"] = df_uid_ts["ds"].values
        fcst_aa["ds"] = df_uid_ts["ds"].values
        # TODO override values
        fcst_tsfm1["ds"] = df_uid_ts["ds"].values

    uid_test = df_uid_ts.merge(fcst_ma, on=["unique_id", "ds"])
    uid_test = uid_test.merge(fcst_aa, on=["unique_id", "ds"])
    # TODO add to uid_test
    uid_test = uid_test.merge(fcst_tsfm1, on=["unique_id", "ds"])


    err = mase(
        df=uid_test, models=model_names, seasonality=seas_len, train_df=df_uid_tr
    )

    pprint(err)

    predictions.append(uid_test)
    results.append(err)
    results_df = pd.concat(results, ignore_index=True)
    print(results_df.mean(numeric_only=True))
    print(results_df.median(numeric_only=True))

results_df = pd.concat(results, ignore_index=True)
predictions_df = pd.concat(predictions, ignore_index=True)
print(results_df.mean(numeric_only=True))
print(results_df.median(numeric_only=True))

T000000
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive  Chronos2
0   T000000   1.372374   1.286285       1.649169  1.371441
MetaARIMA        1.372374
AutoARIMA        1.286285
SeasonalNaive    1.649169
Chronos2         1.371441
dtype: float64
MetaARIMA        1.372374
AutoARIMA        1.286285
SeasonalNaive    1.649169
Chronos2         1.371441
dtype: float64
T000001
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive  Chronos2
0   T000001   0.943842   0.901422       1.249447  0.886982
MetaARIMA        1.158108
AutoARIMA        1.093853
SeasonalNaive    1.449308
Chronos2         1.129212
dtype: float64
MetaARIMA        1.158108
AutoARIMA        1.093853
SeasonalNaive    1.449308
Chronos2         1.129212
dtype: float64
T000002
  unique_id  MetaARIMA  AutoARIMA  SeasonalNaive  Chronos2
0   T000002   0.734334   1.561805       0.256721  0.205779
MetaARIMA        1.016850
AutoARIMA        1.249837
SeasonalNaive    1.051779
Chronos2         0.821401
dtype: float64
MetaARIMA        0.943842
A

In [4]:
output_dir = "assets/results/chronos2"
os.makedirs(output_dir, exist_ok=True)

In [5]:
results_df.to_csv(f"assets/results/chronos2/scores,{target}.csv", index=False)
predictions_df.to_csv(f"assets/results/chronos2/predictions,{target}.csv", index=False)